# XACLE 2027 — D02 MOE8 Gate Entropy Diagnostics

A read-only diagnostic of the frozen MOE8 gate on official train and validation data. It measures gate entropy and routing by score bin and generator, then compares learned gating with uniform averaging and hard top-1 routing. The analysis checks for score compression and prediction errors and provides evidence for deciding whether G01 is worth a follow-up.

The gate receives only the pinned Full8 feature vector. Labels and generator metadata are joined after frozen predictions. Validation and optional public-test labels are evaluation-only. No private evaluation data is attached or accessed. D02 is descriptive and does not establish causality.


In [ ]:
# 2. Runtime configuration
from pathlib import Path
import json
import os
import subprocess
import sys
from datetime import datetime, timezone

RUN_D02 = True
PROFILE = "full_8"
RUN_TRAIN_ANALYSIS = True
RUN_VALIDATION_ANALYSIS = True
RUN_PUBLIC_TEST_ANALYSIS = False
REUSE_EXISTING_CACHE = True
ALLOW_MISSING_EXPERT_EXTRACTION = True
TRAIN_GATE = False
TRAIN_EXPERTS = False
BATCH_SIZE_DIAGNOSTICS = 128
BATCH_SIZE_E8_EXTRACTION = 1
OUTPUT_ROOT = Path("/kaggle/working/d02_moe8")
D02_SOURCE_COMMIT = "330fc7106ee76d6151ec2cd5380eb77d35146a89"
REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
REPO_BRANCH = "research/xacle2027-d02-moe8-gate-diagnostics"
REPO_ROOT = Path("/kaggle/working/moescore_d02_source")
BOOTSTRAP_SAMPLES = 500
RANDOM_SEED = 2027
HIGH_SCORE_ACCURACY_TOLERANCE = 0.5
E8_EXTRACTION_DEVICE = "not_run"

assert RUN_D02 and PROFILE == "full_8"
assert TRAIN_GATE is False and TRAIN_EXPERTS is False
assert BATCH_SIZE_E8_EXTRACTION == 1
for folder in ("diagnostics", "figures", "reports", "logs"):
    (OUTPUT_ROOT / folder).mkdir(parents=True, exist_ok=True)
EXECUTION_LOG = OUTPUT_ROOT / "logs" / "execution.log"
EXECUTION_LOG.write_text("", encoding="utf-8")

def log_event(event, **details):
    record = {"time_utc": datetime.now(timezone.utc).isoformat(), "event": event, **details}
    with EXECUTION_LOG.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, sort_keys=True, default=str) + "\n")
    print(event, details)

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, sort_keys=True,
                                     allow_nan=False, default=str) + "\n",
                          encoding="utf-8")

log_event("configuration", profile=PROFILE, train_gate=TRAIN_GATE,
          train_experts=TRAIN_EXPERTS, public_test=RUN_PUBLIC_TEST_ANALYSIS,
          source_commit=D02_SOURCE_COMMIT)


In [ ]:
# 3. Pin source and check core dependencies
import importlib.util

if len(D02_SOURCE_COMMIT) != 40:
    raise ValueError("D02_SOURCE_COMMIT must be a full Git commit SHA")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--branch", REPO_BRANCH,
                    "--single-branch", REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth=1", "origin",
                D02_SOURCE_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach",
                D02_SOURCE_COMMIT], check=True)
source_sha = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
if source_sha != D02_SOURCE_COMMIT:
    raise RuntimeError(f"Source pin mismatch: expected {D02_SOURCE_COMMIT}, got {source_sha}")
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
missing = [name for name in ("torch", "numpy", "pandas", "scipy", "matplotlib")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Kaggle runtime is missing required D02 packages: {missing}")
import numpy as np
import pandas as pd
import torch
from model.expert_adapters.registry import profile_contract
from training.checkpoint_manager import full8_source_code_sha256
from data.moe_feature_store import validate_feature_store

EXPERT_ORDER, EXPERT_DIMS, FEATURE_WIDTH = profile_contract(PROFILE)
if tuple(EXPERT_ORDER) != tuple(f"expert{i}" for i in range(1, 9)) or FEATURE_WIDTH != 33032:
    raise RuntimeError("Pinned profile_contract('full_8') violates the MOE8 contract")
log_event("source_preflight_pass", source_commit=source_sha, feature_width=FEATURE_WIDTH,
          expert_order=list(EXPERT_ORDER), torch=torch.__version__)


In [ ]:
# 4. Resolve the exact Kaggle inputs and official split metadata
import hashlib

KAGGLE_INPUT = Path("/kaggle/input")
DATASET_REFS = {
    "gate": "qnv8071/moe8-bestgate",
    "recovery": "qnv8071/recovery",
    "weights": "qnv8071/xacle-weights",
    "official_data": "qnv8071/xacle-2027-data",
    "full9_cache": "qnv8071/moescore-full9-feature-cache",
    "hf_text_cache": "qnv8071/moescore-hf-text-cache-v2",
}
DATASET_VERSIONS_AT_UPLOAD = {
    "qnv8071/moe8-bestgate": 1,
    "qnv8071/recovery": 1,
    "qnv8071/xacle-weights": 3,
    "qnv8071/xacle-2027-data": 1,
    "qnv8071/moescore-full9-feature-cache": 1,
    "qnv8071/moescore-hf-text-cache-v2": 12,
}

def find_dataset_root(ref):
    owner = ref.split("/", 1)[0]
    slug = ref.rstrip("/").split("/")[-1]
    matches = [path.resolve() for path in KAGGLE_INPUT.rglob(slug)
               if path.is_dir() and path.name == slug]
    found = [path for path in matches if path.parent.name == owner]
    if not found:
        found = [path for path in matches if path.parent == KAGGLE_INPUT]
    if not found:
        found = [path for path in matches if owner in path.parts]
    found = sorted(set(found))
    if len(found) != 1:
        available = sorted(path.name for path in KAGGLE_INPUT.iterdir() if path.is_dir())
        raise FileNotFoundError(f"Expected one mount for {owner}/{slug} from {ref}; found {found}; available={available}")
    return found[0]

if not KAGGLE_INPUT.is_dir():
    raise FileNotFoundError("/kaggle/input is unavailable")
INPUTS = {key: find_dataset_root(ref) for key, ref in DATASET_REFS.items()}

def official_root(root):
    candidates = [p for p in root.rglob("XACLE_Challenge_2027_development_dataset")
                  if p.is_dir()]
    if (root / "wav").is_dir() and (root / "meta_data").is_dir():
        candidates.append(root)
    valid = sorted(set(p.resolve() for p in candidates
                       if (p / "wav").is_dir() and (p / "meta_data").is_dir()))
    if len(valid) != 1:
        raise ValueError(f"Expected one official data directory; found {valid}")
    return valid[0]

DATA_ROOT = official_root(INPUTS["official_data"])
EXPECTED_COUNTS = {"train": 7500, "validation": 3000, "test": 3000}
METADATA_PATHS = {
    split: DATA_ROOT / "meta_data" / f"{split}_average.csv"
    for split in EXPECTED_COUNTS
}
metadata_identity = {}
for split, expected in EXPECTED_COUNTS.items():
    if not (DATA_ROOT / "wav" / split).is_dir() or not METADATA_PATHS[split].is_file():
        raise FileNotFoundError(f"Official {split} audio or metadata is missing")
    columns = pd.read_csv(METADATA_PATHS[split], nrows=0).columns.tolist()
    filename_column = next((name for name in (
        "wave_file_name", "wav_file_name", "filename", "file_name"
    ) if name in columns), None)
    if filename_column is None:
        raise ValueError(f"{split} metadata has no filename column")
    filenames = pd.read_csv(METADATA_PATHS[split], usecols=[filename_column])[filename_column].astype(str)
    if len(filenames) != expected:
        raise ValueError(f"{split} official sample count or identity is invalid")
    metadata_identity[split] = {
        "path": str(METADATA_PATHS[split]), "filename_column": filename_column,
        "sample_count": int(len(filenames)),
        "filename_sha256": hashlib.sha256(
            "\n".join(filenames.tolist()).encode("utf-8")
        ).hexdigest(),
    }
write_json(OUTPUT_ROOT / "logs/input_mapping.json", {
    "refs": DATASET_REFS, "versions_at_upload": DATASET_VERSIONS_AT_UPLOAD,
    "paths": {key: str(value) for key, value in INPUTS.items()},
    "official_metadata": metadata_identity,
})
log_event("input_discovery_pass", refs=DATASET_REFS, versions=DATASET_VERSIONS_AT_UPLOAD,
          official_counts={s: v["sample_count"] for s, v in metadata_identity.items()})


In [ ]:
# 5. Validate the frozen gate checkpoint and source compatibility
from scripts.diagnostics.d01_moe8 import (
    FULL8_INFERENCE_COMPAT_SOURCE_FILES, prepare_format7_gate_payload,
)
from research.xacle27.analysis.d02_gate_diagnostics import validate_checkpoint_header

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

gate_files = sorted(INPUTS["gate"].rglob("gate8_best.pt"))
if len(gate_files) != 1:
    raise FileNotFoundError(f"Expected one gate8_best.pt; found {gate_files}")
GATE_CHECKPOINT = gate_files[0].resolve()
checkpoint_sha256 = file_sha256(GATE_CHECKPOINT)
payload = torch.load(GATE_CHECKPOINT, map_location="cpu", weights_only=True)
validate_checkpoint_header(payload)
if payload.get("format_version") != 7:
    raise ValueError(f"Expected Full8 format 7; got {payload.get('format_version')}")
if payload.get("expert_feature_dimensions") != dict(EXPERT_DIMS):
    raise ValueError("Checkpoint feature dimensions differ from Full8 profile_contract")
expected_shapes = {
    "net.0.weight": [512, 33032], "net.0.bias": [512],
    "net.1.weight": [512], "net.1.bias": [512],
    "net.4.weight": [8, 512], "net.4.bias": [8],
}
state_shapes = {k: list(v.shape) for k, v in payload["gating_state_dict"].items()}
if state_shapes != expected_shapes:
    raise ValueError(f"MOE8 gate state-dictionary shapes mismatch: {state_shapes}")
source_hashes = full8_source_code_sha256()
recorded_source = payload.get("source_code_sha256", {})
changed_critical = [name for name in FULL8_INFERENCE_COMPAT_SOURCE_FILES
                    if recorded_source.get(name) != source_hashes.get(name)]
if changed_critical:
    raise ValueError(f"Checkpoint inference source compatibility failed: {changed_critical}")
_, _, source_audit, _ = prepare_format7_gate_payload(
    payload, current_source_hashes=source_hashes
)
if payload["config"].get("data", {}).get("profile") != PROFILE:
    raise ValueError("Gate training config is not Full8")
CHECKPOINT_META = {
    "format_version": payload["format_version"], "epoch": int(payload["epoch"]),
    "global_step": int(payload["global_step"]), "best_srcc": float(payload["best_srcc"]),
    "expert_order": payload["expert_order"],
    "expert_feature_dimensions": payload["expert_feature_dimensions"],
    "gate_architecture": payload["gate_architecture"],
    "source_code_sha256": payload["source_code_sha256"],
    "source_audit": source_audit,
    "expert_checkpoint_provenance": payload["expert_checkpoint_provenance"],
    "feature_store_identity": payload["feature_store_identity"],
    "training_config": payload.get("training_config", payload["config"]),
    "config": payload["config"],
}
preflight = {
    "status": "PASS", "checkpoint_sha256": checkpoint_sha256,
    "checkpoint_epoch": CHECKPOINT_META["epoch"], "format_version": CHECKPOINT_META["format_version"],
    "gate_state_shapes": state_shapes, "expert_order": CHECKPOINT_META["expert_order"],
    "feature_width": FEATURE_WIDTH, "num_experts": len(EXPERT_ORDER),
    "critical_inference_source_hashes": "PASS",
    "checkpoint_source_file_count": len(recorded_source),
    "checkpoint_source_revision": "The gate stores per-file source hashes, not a Git commit SHA.",
    "source_differences_outside_compatibility_subset":
        len(source_audit["different_or_unavailable_files"]),
    "training_configuration": CHECKPOINT_META["training_config"],
}
write_json(OUTPUT_ROOT / "logs/preflight.json", preflight)
del payload
log_event("checkpoint_preflight_pass", checkpoint_sha256=checkpoint_sha256,
          epoch=CHECKPOINT_META["epoch"], source_compatibility="PASS")


In [ ]:
# 6. Audit existing complete Full8 caches; never coerce Full9
def manifest_ids(manifest):
    return [sid for shard in manifest["shards"] for sid in shard["sample_ids"]]

def verify_full8(root, splits=("train", "validation")):
    root = Path(root).resolve(strict=True)
    result = {}
    identity = CHECKPOINT_META["feature_store_identity"]
    for split in splits:
        m = validate_feature_store(root, split=split, require_labels=split != "test")
        ids = manifest_ids(m)
        if (m.get("expert_profile"), m.get("feature_width"), m.get("score_width"),
            len(ids)) != ("full_8", 33032, 8, EXPECTED_COUNTS[split]):
            raise ValueError(f"{split}: incompatible Full8 dimensions/profile/count")
        if m.get("expert_order") != list(EXPERT_ORDER):
            raise ValueError(f"{split}: expert order mismatch")
        if m.get("checkpoint_provenance") != CHECKPOINT_META["expert_checkpoint_provenance"]:
            raise ValueError(f"{split}: checkpoint/cache provenance mismatch")
        if m["labels_present"] is not (split != "test"):
            raise ValueError(f"{split}: labels violate split contract")
        if any(not sid.startswith(split + ":") for sid in ids):
            raise ValueError(f"{split}: sample IDs do not match split")
        if split in ("train", "validation"):
            if ids != identity["source_sample_ids_by_split"][split]:
                raise ValueError(f"{split}: sample IDs/order mismatch with checkpoint")
            if m["store_identity_sha256"] != identity[split]:
                raise ValueError(f"{split}: store identity mismatch with checkpoint")
            fp = m["assembly_identity"]["ordered_source_fingerprints"]
            expected = identity["source_wav_sha256_by_split"][split]
            if [row["wav_sha256"] for row in fp] != expected:
                raise ValueError(f"{split}: audio fingerprint order mismatch")
        result[split] = m
    return result

required_splits = ("train", "validation", "test") if RUN_PUBLIC_TEST_ANALYSIS else ("train", "validation")
candidates = set()
for mount in list(INPUTS.values()) + [OUTPUT_ROOT]:
    for mp in mount.rglob("train/manifest.json"):
        try:
            if json.loads(mp.read_text()).get("expert_profile") == "full_8":
                candidates.add(mp.parent.parent.resolve())
        except (OSError, ValueError):
            pass
compatible, rejected = [], []
for root in sorted(candidates):
    try:
        compatible.append((root, verify_full8(root, required_splits)))
    except Exception as exc:
        rejected.append({"path": str(root), "reason": f"{type(exc).__name__}: {exc}"})
if len(compatible) > 1:
    raise ValueError(f"Multiple compatible Full8 stores found: {[str(x[0]) for x in compatible]}")
if REUSE_EXISTING_CACHE and compatible:
    FULL8_ROOT, full8_manifests = compatible[0]
    NEED_FULL8_BUILD = False
else:
    FULL8_ROOT = OUTPUT_ROOT / "full8_store"
    full8_manifests, NEED_FULL8_BUILD = {}, True
cache_audit = {
    "reused_complete_full8_store": not NEED_FULL8_BUILD,
    "candidate_count": len(candidates), "compatible_count": len(compatible),
    "rejected_candidates": rejected, "selected_root": str(FULL8_ROOT) if compatible else None,
    "contract": {"profile": PROFILE, "feature_width": FEATURE_WIDTH, "score_width": 8,
                 "expert_order": list(EXPERT_ORDER)},
}
write_json(OUTPUT_ROOT / "logs/cache_audit.json", cache_audit)
log_event("cache_audit_complete", compatible=len(compatible), candidates=len(candidates),
          reconstruction_needed=NEED_FULL8_BUILD)


In [ ]:
# 7. Reconstruct only missing Full8 features with frozen repository builders
if NEED_FULL8_BUILD:
    import yaml
    reconstruction_missing = [name for name in ("soundfile", "tqdm")
                               if importlib.util.find_spec(name) is None]
    if reconstruction_missing:
        raise RuntimeError(f"Full7/Full8 reconstruction dependencies are missing: {reconstruction_missing}")
    from scripts.kaggle_full9_artifacts import restore_feature_cache, validate_full9_store
    from data.moe_expert_cache import sha256_file as source_sha256
    from scripts.extract_expert8_features import validate_expert8_cache_manifest

    def find_one_file(root, name):
        found = sorted(p.resolve() for p in Path(root).rglob(name) if p.is_file())
        if len(found) != 1:
            raise FileNotFoundError(f"Expected one {name}; found {found}")
        return found[0]

    full9_roots = set()
    for mp in INPUTS["full9_cache"].rglob("train/manifest.json"):
        try:
            body = json.loads(mp.read_text())
        except (OSError, ValueError):
            continue
        root = mp.parent.parent
        if body.get("expert_profile") == "full_9" and all(
            (root / s / "manifest.json").is_file() for s in EXPECTED_COUNTS
        ):
            full9_roots.add(root.resolve())
    if not full9_roots:
        archives = list(INPUTS["full9_cache"].rglob("upload_manifest.json"))
        if len(archives) != 1:
            raise FileNotFoundError("No complete Full9 store or unique uploaded cache archive")
        restored = OUTPUT_ROOT / "full9_restore"
        if not (restored / "feature_cache").exists():
            restore_feature_cache(archives[0].parent, restored)
        for mp in (restored / "feature_cache").rglob("train/manifest.json"):
            try:
                body = json.loads(mp.read_text())
            except (OSError, ValueError):
                continue
            root = mp.parent.parent
            if body.get("expert_profile") == "full_9" and all(
                (root / s / "manifest.json").is_file() for s in EXPECTED_COUNTS
            ):
                full9_roots.add(root.resolve())
    if len(full9_roots) != 1:
        raise ValueError(f"Expected one compatible Full9 source store; found {full9_roots}")
    FULL9_ROOT = next(iter(full9_roots))
    validate_full9_store(FULL9_ROOT, EXPECTED_COUNTS)

    e5 = find_one_file(INPUTS["recovery"], "expert5_best.pt")
    e6 = find_one_file(INPUTS["recovery"], "expert6_best.pt")
    provenance = CHECKPOINT_META["expert_checkpoint_provenance"]
    if source_sha256(e5) != provenance["expert5"]["checkpoint_sha256"]:
        raise ValueError("Frozen E5 checkpoint SHA does not match gate provenance")
    if source_sha256(e6) != provenance["expert6"]["checkpoint_sha256"]:
        raise ValueError("Frozen E6 checkpoint SHA does not match gate provenance")
    whisper_roots = [p.resolve() for p in INPUTS["recovery"].rglob("whisper_cache")
                     if all((p / s / "manifest.json").is_file()
                            for s in ("train", "validation", "test"))]
    if len(whisper_roots) != 1:
        raise FileNotFoundError(f"Expected one complete Whisper cache; found {whisper_roots}")
    WHISPER_ROOT = whisper_roots[0]
    overlap_allowed = bool(CHECKPOINT_META["config"]["data"].get(
        "allow_train_validation_audio_overlap", False))
    build_splits = ["train", "validation"] + (["test"] if RUN_PUBLIC_TEST_ANALYSIS else [])
    FULL7_ROOT = OUTPUT_ROOT / "full7_store"
    cmd = [sys.executable, "scripts/build_full7_feature_store.py",
           "--full9-root", str(FULL9_ROOT), "--whisper-root", str(WHISPER_ROOT),
           "--e5-checkpoint", str(e5), "--e6-checkpoint", str(e6),
           "--source-audio-root", str(DATA_ROOT / "wav"), "--output-root", str(FULL7_ROOT)]
    for split in build_splits:
        cmd += ["--split", split]
    if overlap_allowed:
        cmd.append("--allow-train-validation-audio-overlap")
    with (OUTPUT_ROOT / "logs/full7_build.log").open("w") as log:
        subprocess.run(cmd, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, check=True)
    for split in build_splits:
        validate_feature_store(FULL7_ROOT, split=split, require_labels=split != "test")

    expected_e8 = provenance["expert8"]
    reusable = []
    for mount in list(INPUTS.values()) + [OUTPUT_ROOT]:
        for train_mp in mount.rglob("expert8/*/train/manifest.json"):
            cache_root, cache_id = train_mp.parents[3].resolve(), train_mp.parents[1].name
            try:
                for split in ("train", "validation"):
                    m = json.loads((cache_root / "expert8" / cache_id / split / "manifest.json").read_text())
                    ci, rows = m["expert_cache_identity"], m["source_records"]
                    if ci["checkpoint_provenance"] != expected_e8:
                        raise ValueError("E8 checkpoint identity mismatch")
                    full7 = validate_feature_store(FULL7_ROOT, split=split, require_labels=True)
                    ids = manifest_ids(full7)
                    names = [n for shard in full7["shards"] for n in shard["wave_file_names"]]
                    fingerprints = full7["assembly_identity"]["ordered_source_fingerprints"]
                    if ([r["sample_id"] for r in rows] != ids
                        or [r["wave_file_name"] for r in rows] != names
                        or [r["wav_sha256"] for r in rows] != [r["wav_sha256"] for r in fingerprints]
                        or [r["caption_sha256"] for r in rows] != [r["caption_sha256"] for r in fingerprints]):
                        raise ValueError("E8 sample/audio/caption rows mismatch")
                    validate_expert8_cache_manifest(m, split=split, expected_records=rows,
                                                    cache_identity=ci)
                reusable.append((cache_root, cache_id))
            except Exception:
                continue
    if len(reusable) > 1:
        raise ValueError(f"Multiple exact E8 caches found: {reusable}")
    if reusable:
        E8_CACHE_ROOT, E8_CACHE_ID = reusable[0]
        log_event("reuse_e8_cache", root=str(E8_CACHE_ROOT), cache_id=E8_CACHE_ID)
    else:
        if not ALLOW_MISSING_EXPERT_EXTRACTION:
            raise FileNotFoundError("No verified E8 cache; extraction is disabled")
        e8_missing = [name for name in ("timm", "einops", "nnAudio", "transformers")
                      if importlib.util.find_spec(name) is None]
        if e8_missing:
            raise RuntimeError(f"E8 extraction dependencies are missing: {e8_missing}")
        e8_base = find_one_file(INPUTS["weights"], "checkpoint-30.pth")
        e8_lora = find_one_file(INPUTS["weights"], "m2d-lora-Nam.pt")
        if source_sha256(e8_base) != expected_e8["base_checkpoint"]["sha256"]:
            raise ValueError("M2D base SHA does not match gate provenance")
        if source_sha256(e8_lora) != expected_e8["finetuned_checkpoint"]["sha256"]:
            raise ValueError("E8 LoRA/CORAL SHA does not match gate provenance")
        models = list(INPUTS["hf_text_cache"].rglob("models--google-bert--bert-base-uncased"))
        if len(models) != 1:
            raise FileNotFoundError(f"Expected one pinned BERT snapshot; found {models}")
        HF_HUB_CACHE = models[0].parent.resolve()
        os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
        os.environ["HF_HOME"] = str(HF_HUB_CACHE.parent)
        from scripts.extract_moe_feature_store import _text_snapshot_identity
        bert_identity = _text_snapshot_identity(
            "google-bert/bert-base-uncased", {"cache_dir": str(HF_HUB_CACHE)}, {})
        if not bert_identity["cache_views"][0]["snapshots"]:
            raise ValueError("Pinned BERT snapshot was not verified")
        if not torch.cuda.is_available():
            raise RuntimeError("Missing E8 features require a Kaggle GPU; choose T4")
        gpu_name = torch.cuda.get_device_name(0)
        if "T4" not in gpu_name.upper():
            raise RuntimeError(f"Expected T4 for E8 extraction; found {gpu_name}")
        E8_EXTRACTION_DEVICE = "cuda:0"
        E8_CACHE_ROOT = OUTPUT_ROOT / "expert8_cache"
        config = yaml.safe_load(
            (REPO_ROOT / "configs/expert8_extraction.example.yaml").read_text())
        config["runtime"].update({"device": "cuda:0", "offline": True,
                                  "cache_dir": str(HF_HUB_CACHE)})
        config["experts"]["expert8"]["base_checkpoint"] = str(e8_base)
        config["experts"]["expert8"]["finetuned_checkpoint"] = str(e8_lora)
        config_path = OUTPUT_ROOT / "expert8_extraction.yaml"
        config_path.write_text(yaml.safe_dump(config, sort_keys=False))
        extraction_report = OUTPUT_ROOT / "logs/expert8_extraction.json"
        cmd = [sys.executable, "scripts/extract_expert8_features.py",
               "--config", str(config_path), "--dataset-root", str(DATA_ROOT),
               "--output-root", str(E8_CACHE_ROOT), "--device", "cuda:0",
               "--batch-size", str(BATCH_SIZE_E8_EXTRACTION), "--shard-size", "8",
               "--report-json", str(extraction_report)]
        with (OUTPUT_ROOT / "logs/expert8_extraction.log").open("w") as log:
            subprocess.run(cmd, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, check=True)
        report = json.loads(extraction_report.read_text())
        if (report.get("status") != "PASS"
            or report.get("base_m2d_checkpoint_sha256") != expected_e8["base_checkpoint"]["sha256"]
            or report.get("lora_checkpoint_sha256") != expected_e8["finetuned_checkpoint"]["sha256"]):
            raise ValueError("E8 extraction report does not match pinned checkpoints")
        E8_CACHE_ID = report["expert_cache_identity"]["cache_id"]
        log_event("e8_extraction_pass", device=gpu_name, batch_size=BATCH_SIZE_E8_EXTRACTION)

    FULL8_ROOT = OUTPUT_ROOT / "full8_store"
    cmd = [sys.executable, "scripts/build_full8_feature_store.py",
           "--full7-root", str(FULL7_ROOT), "--expert8-cache-root", str(E8_CACHE_ROOT),
           "--expert8-cache-id", str(E8_CACHE_ID), "--output-root", str(FULL8_ROOT)]
    for split in build_splits:
        cmd += ["--split", split]
    if overlap_allowed:
        cmd.append("--allow-train-validation-audio-overlap")
    with (OUTPUT_ROOT / "logs/full8_build.log").open("w") as log:
        subprocess.run(cmd, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, check=True)
    full8_manifests = verify_full8(FULL8_ROOT, build_splits)
    NEED_FULL8_BUILD = False
    cache_audit.update({"reconstruction": "frozen repository Full9/Full7/Full8 builders",
                        "full9_root": str(FULL9_ROOT), "full7_root": str(FULL7_ROOT),
                        "expert8_cache_root": str(E8_CACHE_ROOT),
                        "expert8_cache_id": E8_CACHE_ID, "selected_root": str(FULL8_ROOT)})
    write_json(OUTPUT_ROOT / "logs/cache_audit.json", cache_audit)
    log_event("full8_reconstruction_pass", root=str(FULL8_ROOT), splits=build_splits)
else:
    log_event("reuse_full8_store", root=str(FULL8_ROOT))


In [ ]:
# 8. Strict Full8 store and split verification
full8_manifests = verify_full8(FULL8_ROOT, required_splits)
for split, manifest in full8_manifests.items():
    if (manifest["feature_width"], manifest["score_width"]) != (33032, 8):
        raise ValueError(f"{split}: invalid [N,33032] features or [N,8] scores")
    if manifest["labels_present"] is not (split != "test"):
        raise ValueError(f"{split}: label presence violates split contract")
log_event("full8_store_verified",
          counts={s: len(manifest_ids(m)) for s, m in full8_manifests.items()},
          identities={s: m["store_identity_sha256"] for s, m in full8_manifests.items()})


In [ ]:
# 9. Frozen gate inference; no optimizer and no parameter updates
from torch.utils.data import DataLoader
from data.moe_training_dataset import MoeTrainingDataset, collate_moe_features
from model.moe_gating_core import GatingNetwork
from scripts.diagnostics.d01_moe8 import load_gate_checkpoint_for_d01
from research.xacle27.analysis.d02_gate_diagnostics import (
    build_per_sample_frame, infer_gate_weights, validate_full8_arrays, SCORE_BIN_LABELS,
)

gate = GatingNetwork(33032, 8, profile="full_8").cpu()
loaded_gate, _, source_check, source_origin = load_gate_checkpoint_for_d01(
    GATE_CHECKPOINT, gate=gate,
    expected_expert_checkpoint_provenance=full8_manifests["train"]["checkpoint_provenance"],
)
ANALYSIS_SPLITS = (["train"] if RUN_TRAIN_ANALYSIS else [])
ANALYSIS_SPLITS += (["validation"] if RUN_VALIDATION_ANALYSIS else [])
ANALYSIS_SPLITS += (["test"] if RUN_PUBLIC_TEST_ANALYSIS else [])
if "validation" not in ANALYSIS_SPLITS:
    raise ValueError("Validation analysis is required")
per_sample = {}
for split in ANALYSIS_SPLITS:
    dataset = MoeTrainingDataset(str(FULL8_ROOT), split=split, require_labels=split != "test")
    if len(dataset) != EXPECTED_COUNTS[split]:
        raise ValueError(f"{split}: wrong row count")
    expected_ids = CHECKPOINT_META["feature_store_identity"].get(
        "source_sample_ids_by_split", {}).get(split)
    observed_ids = manifest_ids(dataset.manifest)
    if split in ("train", "validation") and observed_ids != expected_ids:
        raise ValueError(f"{split}: dataset row order differs from gate provenance")
    loader = DataLoader(dataset, batch_size=BATCH_SIZE_DIAGNOSTICS, shuffle=False,
                        num_workers=0, collate_fn=collate_moe_features)
    frames = []
    for batch in loader:
        x, scores = batch["features"], batch["scores"]
        weights = infer_gate_weights(gate, x, batch_size=BATCH_SIZE_DIAGNOSTICS,
                                     device="cpu").numpy()
        validate_full8_arrays(x.numpy(), scores.numpy(), weights, expert_order=EXPERT_ORDER)
        labels = batch.get("labels_raw")
        frame = build_per_sample_frame(
            batch["sample_ids"], batch["wave_file_names"], split, scores.numpy(), weights,
            labels_raw=labels.numpy() if split == "validation" and labels is not None else None,
        )
        if split == "train":
            if labels is None:
                raise ValueError("Train labels are missing from the verified store")
            raw = labels.numpy().astype(float)
            if not np.isfinite(raw).all() or np.any((raw < 0) | (raw > 10)):
                raise ValueError("Train labels are not finite raw [0,10] values")
            frame["ground_truth_score"] = raw
            frame["prediction_error"] = frame["predicted_score"].to_numpy() - raw
            frame["absolute_prediction_error"] = np.abs(frame["prediction_error"].to_numpy())
            frame["ground_truth_score_bin"] = [SCORE_BIN_LABELS[min(int(v), 9)] for v in raw]
        frames.append(frame)
    per_sample[split] = pd.concat(frames, ignore_index=True)
    if len(per_sample[split]) != EXPECTED_COUNTS[split]:
        raise ValueError(f"{split}: inference sample coverage mismatch")
    log_event("gate_inference_pass", split=split, count=len(per_sample[split]), device="cpu")
if gate.training or any(p.grad is not None for p in gate.parameters()):
    raise RuntimeError("Gate remained in training mode or accumulated gradients")


In [ ]:
# 10. Entropy and routing summaries
from research.xacle27.analysis.d02_gate_diagnostics import (
    entropy_statistics, gate_weights_by_expert,
)
entropy_summaries = {}
weight_tables = []
for split, rows in per_sample.items():
    weights = rows[[f"gate_weight_{i}" for i in range(1, 9)]].to_numpy()
    stats = entropy_statistics(weights)
    entropy_summaries[split] = {
        "sample_count": int(len(rows)),
        "mean_entropy": float(stats["shannon_entropy"].mean()),
        "median_entropy": float(stats["shannon_entropy"].median()),
        "mean_normalized_entropy": float(stats["normalized_entropy"].mean()),
        "normalized_entropy_std": float(stats["normalized_entropy"].std(ddof=0)),
        "mean_effective_experts": float(stats["effective_experts"].mean()),
        "mean_max_gate_weight": float(stats["max_gate_weight"].mean()),
        "fraction_normalized_entropy_ge_0_90": float((stats["normalized_entropy"] >= .90).mean()),
        "fraction_normalized_entropy_ge_0_95": float((stats["normalized_entropy"] >= .95).mean()),
    }
    summary = gate_weights_by_expert(rows)
    summary.insert(0, "split", split)
    weight_tables.append(summary)
weight_results = pd.concat(weight_tables, ignore_index=True)
weight_results.to_csv(OUTPUT_ROOT / "diagnostics/gate_weights_by_expert.csv", index=False)
log_event("entropy_summary_pass", summaries=entropy_summaries)


In [ ]:
# 11. Score bins and generator metadata (one-to-one filename join)
from research.xacle27.analysis.d02_gate_diagnostics import (
    join_metadata_by_filename, score_bin_summary, generator_summary, SCORE_BIN_LABELS,
)

def read_generator_metadata(split):
    columns = pd.read_csv(METADATA_PATHS[split], nrows=0).columns.tolist()
    filename = next((name for name in
        ("wave_file_name", "wav_file_name", "filename", "file_name") if name in columns), None)
    if filename is None or "model_id" not in columns:
        raise ValueError(f"{split} metadata needs filename and model_id")
    selected = [filename, "model_id"] + (["model_name"] if "model_name" in columns else [])
    return pd.read_csv(METADATA_PATHS[split], usecols=selected)

for split in ("train", "validation"):
    if split in per_sample:
        per_sample[split] = join_metadata_by_filename(
            per_sample[split], read_generator_metadata(split)
        )
if RUN_PUBLIC_TEST_ANALYSIS:
    per_sample["test"] = join_metadata_by_filename(
        per_sample["test"], read_generator_metadata("test"))
    columns = pd.read_csv(METADATA_PATHS["test"], nrows=0).columns.tolist()
    score_names = ("ground_truth_score", "label_raw", "average_score", "mean_score",
                   "human_score", "score", "mos")
    score_column = next((name for name in score_names if name in columns), None)
    if score_column is None:
        raise ValueError("Public-test analysis requires an official label column")
    filename = next(name for name in
        ("wave_file_name", "wav_file_name", "filename", "file_name") if name in columns)
    labels = pd.read_csv(METADATA_PATHS["test"], usecols=[filename, score_column])
    labels["_key"] = labels[filename].map(lambda v: Path(str(v)).name)
    label_map = labels.set_index("_key")[score_column]
    prediction_keys = per_sample["test"]["wave_file_name"].map(lambda v: Path(str(v)).name)
    if set(prediction_keys) != set(label_map.index) or len(prediction_keys) != len(label_map):
        raise ValueError("Public-test labels are not a one-to-one match for predictions")
    rows = per_sample["test"]
    rows["ground_truth_score"] = rows["wave_file_name"].map(
        lambda v: label_map.get(Path(str(v)).name))
    raw = rows["ground_truth_score"].to_numpy(dtype=float)
    if not np.isfinite(raw).all() or np.any((raw < 0) | (raw > 10)):
        raise ValueError("Public-test labels failed the exact join or raw [0,10] check")
    rows["prediction_error"] = rows["predicted_score"] - raw
    rows["absolute_prediction_error"] = np.abs(rows["prediction_error"])
    rows["ground_truth_score_bin"] = [SCORE_BIN_LABELS[min(int(v), 9)] for v in raw]

score_tables, generator_tables = [], []
for split, rows in per_sample.items():
    for table, function in ((score_tables, score_bin_summary),
                            (generator_tables, generator_summary)):
        result = function(rows)
        result.insert(0, "split", split)
        table.append(result)
    rows.to_csv(OUTPUT_ROOT / "diagnostics" / f"{split}_per_sample.csv", index=False)
score_bin_results = pd.concat(score_tables, ignore_index=True)
generator_results = pd.concat(generator_tables, ignore_index=True)
score_bin_results.to_csv(OUTPUT_ROOT / "diagnostics/entropy_by_score_bin.csv", index=False)
generator_results.to_csv(OUTPUT_ROOT / "diagnostics/entropy_by_generator.csv", index=False)
log_event("score_and_generator_analysis_pass", split_counts={
    s: len(rows) for s, rows in per_sample.items()})


In [ ]:
# 12. Matched learned, uniform, and hard top-1 comparison
from research.xacle27.analysis.d02_gate_diagnostics import compare_predictions

tables = []
for split, rows in per_sample.items():
    if rows["ground_truth_score"].isna().any():
        continue
    result = compare_predictions(rows, bootstrap_samples=BOOTSTRAP_SAMPLES,
                                 seed=RANDOM_SEED)
    result.insert(0, "split", split)
    tables.append(result)
if not tables:
    raise RuntimeError("No labeled split is available for comparison")
control_comparison = pd.concat(tables, ignore_index=True)
control_comparison.to_csv(OUTPUT_ROOT / "diagnostics/control_comparison.csv", index=False)
validation_comparison = control_comparison.loc[
    control_comparison["split"] == "validation"].copy()
if not {"learned", "uniform", "hard_top1"}.issubset(
    set(validation_comparison["prediction_method"])):
    raise ValueError("Validation comparison is missing a required prediction method")
log_event("matched_control_comparison_pass", validation_rows=len(per_sample["validation"]))


In [ ]:
# 13. Publication-quality D02 figures
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

FIGURE_ROOT = OUTPUT_ROOT / "figures"
val = per_sample["validation"].copy()
weight_columns = [f"gate_weight_{i}" for i in range(1, 9)]
expert_labels = list(EXPERT_ORDER)

def save_figure(name):
    plt.tight_layout()
    plt.savefig(FIGURE_ROOT / name, dpi=180, bbox_inches="tight")
    plt.close()

plt.figure(figsize=(7, 4.5))
plt.hist(val["normalized_entropy"], bins=np.linspace(0, 1, 31),
         color="#4472C4", edgecolor="white")
plt.axvline(.90, color="#C00000", linestyle="--", label="0.90 descriptive marker")
plt.axvline(.95, color="#ED7D31", linestyle=":", label="0.95 descriptive marker")
plt.xlabel("Normalized entropy"); plt.ylabel("Samples")
plt.title(f"Validation gate entropy (n={len(val):,})"); plt.legend()
save_figure("entropy_histogram.png")

plt.figure(figsize=(8, 4.5))
plt.bar(expert_labels, val[weight_columns].mean(),
        yerr=val[weight_columns].std(ddof=0), capsize=3)
plt.axhline(1/8, color="#C00000", linestyle="--", label="Uniform reference")
plt.ylabel("Mean weight ± SD"); plt.title("Validation mean gate weights"); plt.legend()
save_figure("mean_gate_weights.png")

matrix = weight_results.loc[
    (weight_results["split"] == "validation") &
    (weight_results["group_type"] == "score_bin")
].pivot(index="expert", columns="group_value", values="mean_weight")
from research.xacle27.analysis.d02_gate_diagnostics import SCORE_BIN_LABELS
matrix = matrix.reindex(index=expert_labels, columns=list(SCORE_BIN_LABELS))
plt.figure(figsize=(11, 5.5))
plt.imshow(matrix.to_numpy(), aspect="auto", cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="Mean gate weight")
bin_counts = score_bin_results.loc[score_bin_results["split"] == "validation"].set_index("ground_truth_bin")["sample_count"].to_dict()
heatmap_labels = [f"{name}\\nn={bin_counts.get(name, 0)}" for name in matrix.columns]
plt.xticks(range(len(matrix.columns)), heatmap_labels, rotation=35, ha="right")
plt.yticks(range(len(matrix.index)), matrix.index)
plt.xlabel("Ground-truth score bin"); plt.ylabel("Expert")
plt.title("Validation routing by score bin")
save_figure("gate_weight_heatmap.png")

bins = score_bin_results.loc[score_bin_results["split"] == "validation"]
plt.figure(figsize=(9, 4.5))
plt.errorbar(range(len(bins)), bins["mean_normalized_entropy"],
    yerr=bins["entropy_std"] / np.sqrt(bins["sample_count"].clip(lower=1)),
    marker="o", capsize=3)
plt.xticks(range(len(bins)), [f"{name}\\nn={count}" for name, count in zip(bins["ground_truth_bin"], bins["sample_count"])], rotation=35, ha="right")
plt.ylim(0, 1.02); plt.xlabel("Ground-truth score bin")
plt.ylabel("Mean normalized entropy"); plt.title("Entropy by score bin (standard error)")
save_figure("entropy_by_score_bin.png")

groups = [g["normalized_entropy"].to_numpy()
          for _, g in val.groupby("model_id", sort=True)]
group_names = [f"{model_id} (n={len(g)})"
               for model_id, g in val.groupby("model_id", sort=True)]
plt.figure(figsize=(max(9, .42*len(groups)), min(24, max(6, .28*len(groups)))))
if groups:
    plt.boxplot(groups, vert=False, showfliers=False)
    plt.yticks(range(1, len(group_names)+1), group_names, fontsize=7)
plt.xlim(0, 1.02); plt.xlabel("Normalized entropy"); plt.ylabel("Generator (model_id)")
plt.title("Validation entropy distribution by generator")
save_figure("entropy_by_generator.png")

plt.figure(figsize=(9, 5))
hist_bins = np.linspace(0, 10, 41)
for name, col, color in (
    ("Learned", "predicted_score", "#4472C4"),
    ("Uniform", "uniform_prediction_score", "#70AD47"),
    ("Hard top-1", "hard_top1_prediction_score", "#ED7D31"),
):
    plt.hist(val[col], bins=hist_bins, histtype="step", linewidth=1.8,
             label=name, color=color)
plt.xlim(0, 10); plt.xlabel("Predicted raw score"); plt.ylabel("Samples")
plt.title("Matched validation score distributions"); plt.legend()
save_figure("score_distributions.png")

plt.figure(figsize=(6.5, 6))
plt.scatter(val["ground_truth_score"], val["predicted_score"],
            s=12, alpha=.35, edgecolors="none")
plt.plot([0, 10], [0, 10], color="#C00000", linestyle="--")
plt.xlim(0, 10); plt.ylim(0, 10)
plt.xlabel("Ground-truth raw score"); plt.ylabel("Learned raw prediction")
plt.title("Validation predicted versus ground truth")
save_figure("score_scatter.png")

plt.figure(figsize=(6.5, 5))
plt.scatter(val["normalized_entropy"], val["absolute_prediction_error"],
            s=12, alpha=.35, edgecolors="none")
plt.xlabel("Normalized entropy"); plt.ylabel("Absolute prediction error")
plt.title(f"Validation entropy versus error (n={len(val):,})")
save_figure("entropy_vs_error.png")

plt.figure(figsize=(6.5, 5))
plt.scatter(val["max_gate_weight"], val["absolute_prediction_error"],
            s=12, alpha=.35, edgecolors="none")
plt.xlabel("Maximum gate weight"); plt.ylabel("Absolute prediction error")
plt.title(f"Validation maximum weight versus error (n={len(val):,})")
save_figure("max_weight_vs_error.png")

high = val.loc[val["ground_truth_score"] >= 9]
accurate = high.loc[high["absolute_prediction_error"] <= HIGH_SCORE_ACCURACY_TOLERANCE]
under = high.loc[high["predicted_score"] < 9]
plt.figure(figsize=(9, 5))
for name, group in (
    (f"True 9–10 (n={len(high)})", high),
    (f"Accurate within 0.5 (n={len(accurate)})", accurate),
    (f"Predicted below 9 (n={len(under)})", under),
):
    if len(group):
        plt.plot(expert_labels, group[weight_columns].mean(), marker="o", label=name)
plt.axhline(1/8, color="#777777", linestyle=":", label="Uniform reference")
plt.xlabel("Expert"); plt.ylabel("Mean gate weight")
plt.title(f"High-score validation routing (n={len(high)})"); plt.legend()
save_figure("high_score_routing.png")
log_event("figures_written", count=len(list(FIGURE_ROOT.glob("*.png"))),
          high_score_samples=len(high), generators=len(groups))


In [ ]:
# 14. Observed D02 summary and conditional G01 follow-up screen
def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

val_entropy = entropy_summaries["validation"]
metric_rows = validation_comparison.set_index("prediction_method")
learned, uniform, hard = metric_rows.loc["learned"], metric_rows.loc["uniform"], metric_rows.loc["hard_top1"]
if val["normalized_entropy"].nunique() > 1 and val["absolute_prediction_error"].nunique() > 1:
    assoc = spearmanr(val["normalized_entropy"], val["absolute_prediction_error"])
    entropy_error_rho, entropy_error_p = float(assoc.statistic), float(assoc.pvalue)
else:
    entropy_error_rho, entropy_error_p = None, None
label_std = float(val["ground_truth_score"].std(ddof=0))
compression = {
    "ground_truth_std": label_std,
    "learned_prediction_std": float(learned["predicted_std"]),
    "uniform_prediction_std": float(uniform["predicted_std"]),
    "hard_top1_prediction_std": float(hard["predicted_std"]),
    "learned_to_ground_truth_std_ratio":
        float(learned["predicted_std"] / label_std) if label_std else None,
}
flags = {
    "majority_at_or_above_0_90_entropy_marker":
        val_entropy["fraction_normalized_entropy_ge_0_90"] > .5,
    "learned_output_narrower_than_hard_top1":
        float(learned["predicted_std"]) < float(hard["predicted_std"]),
    "positive_entropy_error_association_p_below_0_05":
        entropy_error_rho is not None and entropy_error_rho > 0
        and entropy_error_p is not None and entropy_error_p < .05,
    "learned_mae_not_better_than_hard_top1":
        float(learned["mae"]) >= float(hard["mae"]),
}
g01_screen = ("D02 supports considering G01 as a follow-up, subject to research review."
              if all(flags.values()) else
              "D02 does not show the full descriptive pattern used here to prioritize G01.")
summary = clean_json({
    "experiment_id": "D02-MOE8", "profile": PROFILE,
    "observed_validation_entropy": val_entropy,
    "observed_validation_control_metrics": validation_comparison.to_dict(orient="records"),
    "observed_validation_compression": compression,
    "observed_entropy_error_spearman": {"rho": entropy_error_rho, "p_value": entropy_error_p},
    "observed_high_score_groups": {
        "true_score_9_to_10_count": int(len(high)),
        "underpredicted_below_9_count": int(len(under)),
        "accurate_within_error_0_5_count": int(len(accurate)),
        "underpredicted_mean_normalized_entropy":
            float(under["normalized_entropy"].mean()) if len(under) else None,
        "accurate_mean_normalized_entropy":
            float(accurate["normalized_entropy"].mean()) if len(accurate) else None,
    },
    "g01_screen_flags": flags, "g01_follow_up_screen": g01_screen,
    "interpretation": {
        "observations": "Matched-sample descriptive diagnostics.",
        "hypothesis": "High entropy may accompany score compression or larger prediction errors.",
        "causality": "D02 cannot establish that entropy causes score compression or error.",
        "thresholds": "0.90 and 0.95 are descriptive markers, not universal failure cutoffs.",
        "g01_decision": "A follow-up screen, not model selection.",
    },
    "sample_counts": {split: len(rows) for split, rows in per_sample.items()},
    "public_test_analysis": RUN_PUBLIC_TEST_ANALYSIS,
})
write_json(OUTPUT_ROOT / "reports/d02_summary.json", summary)

metric_cols = ["prediction_method", "srcc", "lcc", "kendall_tau_b", "mse",
               "rmse", "mae", "predicted_std", "srcc_ci_95_low",
               "srcc_ci_95_high", "mae_ci_95_low", "mae_ci_95_high"]
table = ["| " + " | ".join(metric_cols) + " |",
         "| " + " | ".join(["---"] * len(metric_cols)) + " |"]
for _, row in validation_comparison.iterrows():
    values = ["undefined" if pd.isna(row[c]) else
              (str(row[c]) if c == "prediction_method" else f"{float(row[c]):.4f}")
              for c in metric_cols]
    table.append("| " + " | ".join(values) + " |")
report = "\n".join([
    "# D02-MOE8 gate entropy diagnostics", "",
    "## Observed results", "",
    f"- Validation samples: {len(val)}.",
    f"- Mean normalized entropy: {val_entropy['mean_normalized_entropy']:.4f}.",
    f"- Fraction at normalized entropy >= 0.90: {val_entropy['fraction_normalized_entropy_ge_0_90']:.4f}.",
    f"- Fraction at normalized entropy >= 0.95: {val_entropy['fraction_normalized_entropy_ge_0_95']:.4f}.",
    f"- Entropy/error Spearman rho: {entropy_error_rho}.",
    f"- G01 follow-up screen: {g01_screen}", "",
    "### Matched validation controls", "", *table, "",
    "## Interpretation", "",
    "A wider output range alone does not establish improved accuracy. This screen is descriptive, not causal. The label-based oracle in validation rows is marked ORACLE_DIAGNOSTIC_ONLY and was not used for inference, training, or selection.", "",
    "## Reproducibility", "",
    f"- Source commit: {D02_SOURCE_COMMIT}",
    f"- Gate SHA-256: {checkpoint_sha256}",
    f"- Gate epoch: {CHECKPOINT_META['epoch']}",
    f"- Train store identity: {full8_manifests['train']['store_identity_sha256']}",
    f"- Validation store identity: {full8_manifests['validation']['store_identity_sha256']}",
    f"- Public-test analysis: {RUN_PUBLIC_TEST_ANALYSIS}",
])
(OUTPUT_ROOT / "reports/d02_report.md").write_text(report + "\n", encoding="utf-8")
write_json(OUTPUT_ROOT / "reports/d02_summary.json", summary)
log_event("summary_written", g01_screen=g01_screen, flags=flags)
print(report)


In [ ]:
# 15. Export the artifact and provenance manifest
manifest_path = OUTPUT_ROOT / "reports/artifact_manifest.json"
artifact_paths = sorted(
    str(path.relative_to(OUTPUT_ROOT))
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file() and path != manifest_path
)
manifest = {
    "experiment_id": "D02-MOE8", "profile": PROFILE,
    "source_commit": D02_SOURCE_COMMIT,
    "checkpoint_sha256": checkpoint_sha256,
    "checkpoint_epoch": CHECKPOINT_META["epoch"],
    "checkpoint_format_version": CHECKPOINT_META["format_version"],
    "checkpoint_source_hash_compatibility": {
        "critical_inference_files": "PASS",
        "checkpoint_source_revision": "No commit ID is encoded; per-file hashes remain authoritative.",
        "recorded_source_file_count": preflight["checkpoint_source_file_count"],
        "different_files_outside_compatibility_subset":
            preflight["source_differences_outside_compatibility_subset"],
    },
    "expert_order": list(EXPERT_ORDER), "feature_width": FEATURE_WIDTH, "num_experts": 8,
    "feature_store_identity": {
        split: {"store_identity_sha256": m["store_identity_sha256"],
                "source_manifest_sha256": m["source_manifest_sha256"]}
        for split, m in full8_manifests.items()
    },
    "cache_provenance": {
        "expert_checkpoint_provenance": CHECKPOINT_META["expert_checkpoint_provenance"],
        "reused_complete_full8_store": not NEED_FULL8_BUILD,
        "cache_audit": cache_audit,
    },
    "dataset_identity": {
        "refs": DATASET_REFS, "versions_at_upload": DATASET_VERSIONS_AT_UPLOAD,
        "mounted_paths": {key: str(path) for key, path in INPUTS.items()},
        "official_metadata": metadata_identity,
    },
    "sample_counts": {split: len(rows) for split, rows in per_sample.items()},
    "verified_official_metadata_counts": EXPECTED_COUNTS,
    "execution_device": {
        "gate_diagnostics": "cpu", "expert8_extraction": E8_EXTRACTION_DEVICE,
    },
    "analysis_parameters": {
        "train_analysis": RUN_TRAIN_ANALYSIS, "validation_analysis": RUN_VALIDATION_ANALYSIS,
        "public_test_analysis": RUN_PUBLIC_TEST_ANALYSIS,
        "train_gate": TRAIN_GATE, "train_experts": TRAIN_EXPERTS,
        "diagnostics_batch_size": BATCH_SIZE_DIAGNOSTICS,
        "expert8_extraction_batch_size": BATCH_SIZE_E8_EXTRACTION,
        "entropy_epsilon": 1e-12, "entropy_markers": [0.90, 0.95],
        "bootstrap_samples": BOOTSTRAP_SAMPLES, "random_seed": RANDOM_SEED,
        "high_score_accuracy_tolerance": HIGH_SCORE_ACCURACY_TOLERANCE,
    },
    "artifact_paths": artifact_paths + ["reports/artifact_manifest.json"],
    "execution_status": "completed",
}
write_json(manifest_path, clean_json(manifest))
log_event("artifact_export_complete", artifact_count=len(manifest["artifact_paths"]),
          output_root=str(OUTPUT_ROOT), execution_status="completed")
print("Artifacts:", OUTPUT_ROOT)
print(*manifest["artifact_paths"], sep="\n- ")
